<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/Aula_3_Transformer_Decoder_only_and_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3 - Transformer *decoder-only*

Nesta aula você irá modificar o Transformer *decoder-only* fornecido a seguir.
Observe que em um *decoder-only* não existe:

*   cross-attention
*   encoder separado

e é utilizado com *auto-regressão*.

## Objetivo


## Exercício

Neste exercício você deve:

1.   carregar o seu conjunto de documentos
2.   treinar e usar (ou carregar) um tokenizador
3.   fazer treino de um modelo decoder-only
4.   incluir no loop de treino, inferência usando máxima probabilidade
5.   incluir no loop de treino, inferência usando amostragem com temperatura


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Passo 1: carregar conjunto de documentos

In [18]:
from datasets import load_dataset
import re

###### INSIRA AQUI O CODIGO PARA CARREGAR OS SEUS DOCUMENTOS na lista DOCUMENTOS
#ds = load_dataset("jquigl/imdb-genres") # Exemplo
# ==========================================
# SOLUÇÃO: criar um "ds" compatível manualmente
# ==========================================

ds = {
    "train": [
        {"description": "este filme é muito bom e divertido"},
        {"description": "este filme é ruim e entediante"},
        {"description": "eu gostei muito deste filme"},
        {"description": "a história é interessante e bem escrita"},
        {"description": "a atuação foi fraca e decepcionante"},
        {"description": "o filme tem uma trilha sonora excelente"},
        {"description": "não gostei do final do filme"},
        {"description": "o filme é lento mas bonito"},
        {"description": "a história é simples e emocionante"},
        {"description": "o filme é empolgante e divertido"}
    ]
}

def clean_ascii(text):
    text = text.encode("ascii", errors="ignore").decode()
    return re.sub(r"[^A-Za-z0-9 .,:;!?'\-]", "", text)

# Extract documentos
documentos = [clean_ascii(x["description"]) for x in ds["train"]]
documentos = [t.split(" - ")[0] for t in documentos]   # optional remove year
documentos = [t for t in documentos if len(t) > 0]

#documentos = [ "meu doc favorito 1", "meu doc menos favorito 2"]
print("Total documentos:", len(documentos))
print("Sample:", documentos[:10])

Total documentos: 10
Sample: ['este filme  muito bom e divertido', 'este filme  ruim e entediante', 'eu gostei muito deste filme', 'a histria  interessante e bem escrita', 'a atuao foi fraca e decepcionante', 'o filme tem uma trilha sonora excelente', 'no gostei do final do filme', 'o filme  lento mas bonito', 'a histria  simples e emocionante', 'o filme  empolgante e divertido']


# Passo 2: Carregar ou treinar um tokenizador

In [19]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
import random
import re

##### Insira aqui o código para treinar o seu TOKENIZER
# Defina o seu tokenizador
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = WordLevelTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)
# Treino do tokenizador
tokenizer.train_from_iterator(documentos, trainer)

# Funções auxiliares para transitar entre tokens textuais e ids de tokens
def encode(text):
    ids = tokenizer.encode("[BOS] " + text + " [EOS]").ids
    return torch.tensor(ids, dtype=torch.long)

def decode(ids):
    return tokenizer.decode(ids.tolist())

vocab_size = tokenizer.get_vocab_size()

## Definição do modelo Transformer Decoder-only

In [20]:
class DecoderOnlyTransformer(nn.Module):
    "Implementação de um transformer que tem somente a parte do decoder"
    def __init__(self, vocab_size, d_model=128, n_heads=4, num_layers=3, max_len=64):
        super().__init__()
        self.max_len = max_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)

        h = self.token_emb(x) + self.pos_emb(pos)

        # Máscara causal
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        out = self.decoder(h, h, tgt_mask=mask)
        logits = self.lm_head(out)
        return logits

# Códigos de inferência

## Código de inferência simples: token com maior probabilidade


In [21]:
def max_prob_sampling(logits):
    next_token = logits.argmax(dim=-1)
    return next_token.unsqueeze(0)

## Código de inferência avançada: amostragem com temperatura

In [22]:
import torch

# Inferência (com temperature e top_p)
def sampling(logits, top_p=0.9, top_k=None, temperature=1.0):
    # Ajusta pela temperatura
    logits = logits / temperature

    # Se top_k for especificado, filtra por top_k primeiro
    if top_k is not None:
        # Ensure top_k is not larger than the vocabulary size
        k_to_use = min(top_k, logits.size(-1))
        # Get the top_k values and indices
        v, _ = torch.topk(logits, k_to_use)
        # Set logits of all values smaller than the k-th value to -inf
        logits[logits < v[:, [-1]]] = float('-inf')

    # Ordena os logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

    # Mascara tokens acima do top_p
    mask = cumulative_probs > top_p
    # Garante que ao menos um token permaneça
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False

    filtered_logits = sorted_logits.masked_fill(mask, float('-inf'))
    probs = torch.softmax(filtered_logits, dim=-1)

    # Amostra o token
    sampled_idx = torch.multinomial(probs, num_samples=1)

    # Converte para índice na tabela original
    next_token = sorted_indices[sampled_idx]

    return next_token


def generate(prompt, next_token_function, max_new_tokens=20, top_k=None, top_p=0.9, temperature=1.0):
    model.eval()
    x = encode(prompt).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # pega apenas o último passo

        # Passa os parâmetros de amostragem se a função for top_p_sampling
        if next_token_function == top_p_sampling:
            next_token = next_token_function(logits.squeeze(0), top_k=top_k, top_p=top_p, temperature=temperature)
        else:
            next_token = next_token_function(logits.squeeze(0))

        x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

        if next_token.item() == tokenizer.token_to_id("[EOS]"):
            break

    return decode(x[0])


# 4. Fazer treino de modelo: códigos de treino

In [25]:
# Função auxiliar para gerar batches de exemplos para treino
def sample_batch(batch_size=16, max_len=20):
    batch = random.sample(documentos, batch_size)
    tokenized = [encode(t) for t in batch]

    max_t = min(max(len(x) for x in tokenized), max_len)
    padded = []

    for x in tokenized:
        x = x[:max_t]
        pad_len = max_t - len(x)
        if pad_len > 0:
            x = torch.cat([x, torch.zeros(pad_len, dtype=torch.long)])
        padded.append(x)

    return torch.stack(padded)

################################
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DecoderOnlyTransformer(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
#################################

steps = 10000

for step in range(1, steps + 1):
    model.train()
    batch = sample_batch().to(device)

    logits = model(batch[:, :-1])
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        batch[:, 1:].reshape(-1),
        ignore_index=tokenizer.token_to_id("[PAD]")
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        ppl = torch.exp(loss).item()
        print(f"[step {step}] loss={loss.item():.4f}, ppl={ppl:.2f}")

    if step % 100 == 0:
        print("Generated text:")
        #print(generate("MAXPROB: This movie is a", next_token_function= ... ))
        print("--------------------------------------")

print("Training completed.")

ValueError: Sample larger than population or is negative

# Exercício: controle de temperatura, Top-K e Top-P

Modifique o código a seguir para fazer a visualização de produção de tokens do modelo que você treinou.

In [13]:
# ==========================================
# PASSO 1 — CONJUNTO DE DOCUMENTOS (PORTUGUÊS)
# ==========================================

documentos = [
    "este filme é muito bom e divertido",
    "este filme é ruim e entediante",
    "eu gostei muito deste filme",
    "a história é interessante e bem escrita",
    "a atuação foi fraca e decepcionante",
    "o filme tem uma trilha sonora excelente",
    "não gostei do final do filme",
    "o filme é lento mas bonito",
    "a história é simples e emocionante",
    "o filme é empolgante e divertido"
]

print("Total de documentos:", len(documentos))
print("Exemplo:", documentos[:3])


Total de documentos: 10
Exemplo: ['este filme é muito bom e divertido', 'este filme é ruim e entediante', 'eu gostei muito deste filme']


In [14]:
# ==========================================
# PASSO 2 — TOKENIZADOR
# ==========================================

import torch
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = WordLevelTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

tokenizer.train_from_iterator(documentos, trainer)

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

def encode(texto):
    ids = tokenizer.encode("[BOS] " + texto + " [EOS]").ids
    return torch.tensor(ids, dtype=torch.long)

def decode(ids):
    return tokenizer.decode(ids.tolist())

vocab_size = tokenizer.get_vocab_size()
print("Tamanho do vocabulário:", vocab_size)


Tamanho do vocabulário: 40


In [15]:
# ==========================================
# PASSO 3 — MODELO DECODER-ONLY
# ==========================================

import torch.nn as nn

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, num_layers=3, max_len=64):
        super().__init__()
        self.max_len = max_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        camada = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(camada, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)

        h = self.token_emb(x) + self.pos_emb(pos)

        # Máscara causal (auto-regressão)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        h = self.transformer(h, mask)
        logits = self.lm_head(h)
        return logits


In [16]:
# ==========================================
# PASSO 4 — INFERÊNCIA
# ==========================================

import torch
import torch.nn.functional as F

@torch.no_grad()
def max_prob_sampling(logits):
    return torch.argmax(logits, dim=-1)

@torch.no_grad()
def sampling(logits, temperature=1.0, top_k=20, top_p=0.9):
    logits = logits / temperature

    if top_k is not None:
        valores, _ = torch.topk(logits, top_k)
        limite = valores[-1]
        logits[logits < limite] = -float("inf")

    probs = F.softmax(logits, dim=-1)
    probs_ord, idx_ord = torch.sort(probs, descending=True)
    acumulada = torch.cumsum(probs_ord, dim=-1)

    mascara = acumulada > top_p
    mascara[0] = False
    probs_ord[mascara] = 0
    probs_ord = probs_ord / probs_ord.sum()

    token = torch.multinomial(probs_ord, 1)
    return idx_ord[token]

@torch.no_grad()
def gerar_texto(prompt, modo="max", max_tokens=20, temperature=1.0):
    model.eval()
    x = encode(prompt).unsqueeze(0).to(device)

    for _ in range(max_tokens):
        logits = model(x)[:, -1, :].squeeze(0)

        if modo == "max":
            prox = max_prob_sampling(logits)
        else:
            prox = sampling(logits, temperature=temperature)

        x = torch.cat([x, prox.view(1, 1)], dim=1)

        if prox.item() == EOS_ID:
            break

    return decode(x[0].cpu())


In [17]:
# ==========================================
# PASSO 5 — TREINO
# ==========================================

import random

def gerar_batch(batch_size=4, max_len=32):
    textos = random.sample(documentos, batch_size)
    tokens = [encode(t) for t in textos]

    tam = min(max(len(t) for t in tokens), max_len)
    batch = []

    for t in tokens:
        t = t[:tam]
        if len(t) < tam:
            t = torch.cat([t, torch.full((tam - len(t),), PAD_ID)])
        batch.append(t)

    return torch.stack(batch)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DecoderOnlyTransformer(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(1, 501):
    model.train()
    batch = gerar_batch().to(device)

    logits = model(batch[:, :-1])
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        batch[:, 1:].reshape(-1),
        ignore_index=PAD_ID
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Passo {step} | Loss: {loss.item():.4f}")
        print("Max prob:", gerar_texto("este filme", modo="max"))
        print("Temperatura:", gerar_texto("este filme", modo="sample", temperature=1.2))
        print("-" * 40)

print("Treinamento finalizado.")


Passo 50 | Loss: 0.4713
Max prob: este filme é ruim e divertido
Temperatura: este filme é ruim entediante
----------------------------------------
Passo 100 | Loss: 0.3651
Max prob: este filme muito bom e divertido
Temperatura: este filme muito bom e divertido
----------------------------------------
Passo 150 | Loss: 0.3296
Max prob: este filme muito bom e divertido
Temperatura: este filme deste filme
----------------------------------------
Passo 200 | Loss: 0.3063
Max prob: este filme ruim e entediante
Temperatura: este filme é ruim e divertido
----------------------------------------
Passo 250 | Loss: 0.3014
Max prob: este filme ruim e entediante
Temperatura: este filme ruim e entediante
----------------------------------------
Passo 300 | Loss: 0.3216
Max prob: este filme muito bom e divertido
Temperatura: este filme ruim e entediante
----------------------------------------
Passo 350 | Loss: 0.3138
Max prob: este filme ruim e entediante
Temperatura: este filme ruim e entediante
-

Neste exercício foi implementado e treinado um modelo Transformer do tipo decoder-only, utilizando auto-regressão e máscara causal. Um tokenizador foi treinado a partir de um conjunto simples de documentos em português, e o modelo foi treinado para prever o próximo token em uma sequência. Durante o treinamento, foram demonstradas duas estratégias de inferência: seleção por máxima probabilidade e amostragem com temperatura, evidenciando o comportamento determinístico e estocástico do modelo.